In [14]:
import cv2
import numpy as np
import onnxruntime as ort

# Load ONNX model once (important for performance)
session = ort.InferenceSession("model/traffic_sign_model.onnx")
input_name = session.get_inputs()[0].name


def preprocess_image(image_path):
    """
    Preprocess image exactly like training:
    - grayscale
    - resize to 32x32
    - histogram equalization
    - normalize
    - reshape to (1, 32, 32, 1)
    """
    img = cv2.imread(image_path)

    if img is None:
        raise ValueError("❌ Image not found")

    # Resize
    img = cv2.resize(img, (32, 32))

    # Convert to grayscale
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Histogram equalization
    img = cv2.equalizeHist(img)

    # Normalize
    img = img / 255.0

    # Reshape → (1, 32, 32, 1)
    img = img.reshape(1, 32, 32, 1).astype(np.float32)

    return img


def predict_image(image_path):
    """
    Predict traffic sign class from image
    """
    img = preprocess_image(image_path)

    # Run ONNX model
    outputs = session.run(None, {input_name: img})

    # Get prediction
    predictions = outputs[0]
    class_id = int(np.argmax(predictions))
    confidence = float(np.max(predictions))

    return {"The image shows the traffic sign ":class_id,
            "Confidence":confidence}

In [15]:
img = "test_images/image.png"

predict_image(img)

{'The image shows the traffic sign ': 10, 'Confidence': 0.9891936182975769}